# Lesson 7.1 — The Orchestrator: Coordination as a Stage

Recover is the sixth stage — pure coordination. It consumes detection's localised fault + owner and re-invokes existing layers; no new theory.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, recover, RECOVERY_POLICY)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def kick(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []


### Healthy setup: recover succeeds on the first attempt

In [2]:
r0 = recover(W, W.q.copy())
print('healthy -> success=%s recovered=%s n_attempts=%d' % (r0['success'], r0['recovered'], r0['n_attempts']))
checks.append(r0['success'] and r0['recovered'] is False and r0['n_attempts'] == 1)


healthy -> success=True recovered=False n_attempts=1


### Transient disturbance: the orchestrator re-invokes execution and recovers

In [3]:
r1 = recover(W, W.q.copy(), disturbance=lambda a: kick if a == 0 else None)
print('transient -> success=%s recovered=%s n_attempts=%d' % (r1['success'], r1['recovered'], r1['n_attempts']))
print('  attempt log:', [(a['event'], a['response']) for a in r1['attempts']])
checks.append(r1['success'] and r1['recovered'] is True and r1['n_attempts'] == 2)
# the response was an existing layer call re-invoked (retry-execute), not new theory
checks.append(r1['attempts'][0]['response'] == 'retry-execute')
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


transient -> success=True recovered=True n_attempts=2
  attempt log: [('TRACKING_FAILURE', 'retry-execute')]
3/3 checks passed.
All checks passed.
